In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, TimestampType
from pyspark.sql.functions import from_json, col, window

STORAGE_ACCOUNT_NAME = "stclickstreamstu"
CONTAINER_NAME = "datalake"

BRONZE_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/bronze/events"
SILVER_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/events"
CHECKPOINT_PATH_SILVER = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/silver_events"

json_schema = StructType([
    StructField("schema_version", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("timestamp", StringType(), True), 
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("os", StringType(), True),
    StructField("action", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount", IntegerType(), True),
    StructField("utm_source", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("is_bot", BooleanType(), True)
])

print("Đã khởi tạo cấu hình và Schema Silver!")

In [0]:
bronze_stream_df = (spark.readStream
    .format("delta")
    .load(BRONZE_TABLE_PATH)
)

In [0]:
from pyspark.sql.functions import to_timestamp

silver_flattened_df = (bronze_stream_df
    .withColumn("parsed_data", from_json(col("raw_payload"), json_schema))
    .select("parsed_data.*", "ingestion_timestamp")
    
    # Ép kiểu lại timestamp từ String sang đúng định dạng thời gian
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    
    # Cleaning:
    # - Loại bỏ những dòng user_id bị NULL
    # - Loại bỏ những dòng giá bị âm hoặc không phải là số (Do bẫy trong main.py)
    # - Loại bỏ các event nghi vấn là BOT
    .filter(
        (col("user_id").isNotNull()) & 
        (col("price") > 0) & 
        (col("is_bot") == False)
    )
    
    # Deduplication:
    # Dùng event_id làm khóa chính, kết hợp watermark 10 phút để tránh tốn RAM
    .withWatermark("event_timestamp", "10 minutes")
    .dropDuplicates(["event_id", "event_timestamp"])
    
    # Loại bỏ cột trung gian không cần thiết
    .drop("timestamp")
)

In [0]:
print(f"Đang ghi dữ liệu sạch xuống Silver tại: {SILVER_TABLE_PATH}")

query_silver = (silver_flattened_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH_SILVER)
    .trigger(availableNow=True) # Hút sạch mẻ Bronze hiện tại
    .start(SILVER_TABLE_PATH)
)

query_silver.awaitTermination()
print("Đã hoàn tất xử lý lớp Silver!")

In [0]:
df_silver_check = spark.read.format("delta").load(SILVER_TABLE_PATH)

print(f"Tổng số dòng sạch hiện có trong Silver: {df_silver_check.count()}")
display(df_silver_check.limit(10))